# Aula 4 — Notebook 2

## Observabilidade em um workflow multiagente

Com vários agentes, não basta registrar apenas a resposta final. Precisamos saber qual agente foi chamado, qual saída produziu, quais ferramentas foram executadas, qual rota foi tomada e quanto cada etapa consumiu.

A observabilidade também ajuda a avaliar se a arquitetura multiagente está se justificando. Caso o agente revisor quase nunca altere o resultado, por exemplo, talvez ele acrescente custo sem benefício proporcional.


## 1. Logs, métricas e traces não são a mesma coisa

- **Evento ou log:** algo que ocorreu em uma etapa específica.
- **Métrica:** valor numérico agregável, como latência ou tokens.
- **Trace:** sequência conectada de eventos de uma execução completa.

No workflow desta aula, os nomes das etapas deixam explícita a origem dos eventos: `triage_agent`, `execute_tools`, `investigation_agent`, `review_agent` e `action_planner_agent`.


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INCIDENT_AGENT_DIR = PROJECT_ROOT / "target_project/incident_agent"
sys.path.insert(0, str(PROJECT_ROOT))

from shared.aula_4.observability import new_run_id, build_event, summarize_runs

run_id = new_run_id()
event = build_event(run_id, "step_started", "retrieve_evidence", {"incident_id": "INC-1001"})
event

## 2. Eventos estruturados

Uma mensagem como `"deu erro"` é pouco útil. Um evento estruturado deve conter campos previsíveis que possam ser consultados por código.

O exemplo da aula registra:

- `run_id`;
- instante em UTC;
- tipo do evento;
- etapa do workflow;
- payload específico.

O payload pode variar, mas os campos externos permanecem estáveis. Essa estabilidade permite filtrar, contar e comparar eventos sem depender da interpretação de texto livre.

In [ ]:
events = [
    build_event(run_id, "agent_completed", "triage_agent", {"selected_tools": ["search_runbooks"]}),
    build_event(run_id, "tools_completed", "execute_tools", {"documents": 3}),
    build_event(run_id, "agent_completed", "investigation_agent", {"classification": "pipeline_failure"}),
    build_event(run_id, "validation_completed", "deterministic_validation", {"is_valid": True}),
    build_event(run_id, "agent_completed", "review_agent", {"decision": "approved"}),
]

events


## 3. Métricas por etapa e por execução

A latência total não explica onde o sistema gastou tempo. Por isso, registramos a duração de cada etapa separadamente.

Além da latência, uma chamada ao modelo pode expor metadados de uso. Quando o provedor os disponibiliza, registramos:

- tokens de entrada;
- tokens de saída;
- total de tokens;
- custo estimado, desde que os preços estejam configurados.

O custo é uma **estimativa configurável**, e não um valor universal. Preços variam por modelo, provedor, contrato e data. Por isso, o código não fixa um preço como verdade permanente.

In [ ]:
from shared.aula_4.observability import estimate_cost_usd

example_cost = estimate_cost_usd(
    input_tokens=2_500,
    output_tokens=650,
    input_price_per_million=0.0,
    output_price_per_million=0.0,
)
example_cost

## 4. Persistência local do trace

Ao final do workflow, o estado completo é salvo como JSON em `data/runtime_logs`.

Essa escolha tem vantagens didáticas:

- não exige um serviço externo;
- permite abrir e inspecionar cada execução;
- facilita a criação de testes e relatórios;
- evidencia quais dados estamos registrando.

Em uma aplicação real, é necessário avaliar cuidadosamente privacidade, retenção e exposição de dados. Logs não devem armazenar segredos, credenciais ou conteúdo sensível sem necessidade e proteção adequadas.

In [ ]:
log_dir = INCIDENT_AGENT_DIR / "data/runtime_logs"
log_dir.mkdir(parents=True, exist_ok=True)
list(log_dir.glob("*.json"))[:5]

## 5. Agregando múltiplas execuções

Uma execução isolada ajuda na depuração. Um conjunto de execuções permite observar comportamento sistêmico.

A função `summarize_runs` calcula indicadores simples:

- quantidade de execuções;
- taxa de conclusão;
- taxa de revisão humana;
- latência média;
- média de tokens.

Esses indicadores não medem, sozinhos, a qualidade do diagnóstico. Eles precisam ser combinados com avaliações de correção e utilidade discutidas na Aula 3.

In [ ]:
example_records = [
    {"status": "completed", "metrics": {"total_latency_ms": 1200, "total_tokens": 900}},
    {"status": "human_review_required", "metrics": {"total_latency_ms": 1700, "total_tokens": 1100}},
    {"status": "completed", "metrics": {"total_latency_ms": 980, "total_tokens": 850}},
]

summarize_runs(example_records)

## 6. Exercício orientado

Depois de executar vários incidentes, calcule separadamente:

- tokens e latência por agente;
- frequência de `revise` e `human_review` emitida pelo revisor;
- ferramentas selecionadas pela triagem;
- taxa de ações bloqueadas pela política;
- custo do workflow multiagente comparado ao baseline de agente único.

Use os resultados para discutir remoção, combinação ou manutenção de agentes. Observabilidade deve apoiar decisões arquiteturais, não apenas produzir dashboards.
